# Deep MLP

In [1]:
import torch
import torch.nn as nn
# Example usage:
torch.manual_seed(0)

In [2]:
def mse_loss(y, y_hat):
    """
    Args:
        y: the label tensor of shape (batch_size, linear_2_out_features)
        y_hat: the prediction tensor of shape (batch_size, linear_2_out_features)

    Return:
        J: scalar loss
        dJdy_hat: The gradient tensor of shape (batch_size, linear_2_out_features)
    """
    # The torch.F.mse_loss function's default behavior is to use the reduction mode 'mean.'
    # Therefore, all the code was written with respect to that.
    #scalar loss
    loss_for_each_y_outputs=((y-y_hat)**2)
    loss_for_each_observation=loss_for_each_y_outputs.mean(axis=1)
    J=loss_for_each_observation.mean(axis=0) 

    # the gradient tensor of y_hat
    # (1/(y.shape[0]*y.shape[1])) is added  for the reduction mode 'mean.'
    dJdy_hat = (2*(y-y_hat)*(-1)) * (1/(y.shape[0]*y.shape[1]))

    """
    Bana not !!!!!!!!!!
    note that Deep learning frameworks choose to represent gradients as the negative of the actual derivative of the loss with respect to model parameters (or predictions) primarily for two reasons:

    1. Consistency with Optimization Algorithms: By representing gradients as the negative direction of increase in the loss, Many optimization  algorithms can directly use the calculated gradients to update parameters in the direction of minimizing the loss. It simplifies the mathematics of optimization algorithms.

    2. Gradient Descent Interpretation: The convention aligns with the intuitive idea of "gradient descent." In gradient descent, you move in the direction opposite to the gradient (negative gradient) to reach the minimum of a function. By using negative gradients, deep learning frameworks maintain this consistency with the traditional interpretation of gradient descent.
    
    """
    return J,dJdy_hat


In [3]:
def cross_entropy_loss(y, y_hat):
    """
    Args:
        y: the correct class indices tensor of shape (batch_size,)
        y_hat: the prediction tensor of shape (batch_size, linear_2_out_features)

    Return:
        J: scalar loss
        dJdy_hat: The gradient tensor of shape (batch_size, linear_2_out_features)
    """
    """
    Categorical Cross-Entropy Loss Function, also known as Cross-Entropy Loss, Softmax Loss,log loss , is
    a loss function used in multiclass classiﬁcation model training. It applies the Softmax Activation
    Function to a model's output (logits) before applying the Negative Log-Likelihood function.

    Here's a step-by-step guide to writing a function for cross-entropy loss:

    1. Apply softmax to your neural network's raw outputs to get probabilities.
    2. Calculate the log of those probabilities (because the cross-entropy formula involves the logarithm of probabilities).
    3. Select the log probability of the correct class for each example in the batch.
    4. Calculate the negative average of these log probabilities (the negative sign is because we want to minimize the loss).
    

    """


    # 1.  Apply softmax to your neural network's raw outputs to get probabilities. (y_hat)
    # first, for numerical stability 
    #Before applying the exponential function in the softmax function, the maximum value of the inputs is subtracted from each input. 
    #This operation does not change the output of the softmax function but prevents large values from being exponentiated,
    #which helps in avoiding overflow.   
    max_y_hat = torch.max(y_hat, dim=1, keepdim=True).values
    exps = torch.exp(y_hat - max_y_hat)
    softmax = exps / torch.sum(exps, dim=1, keepdim=True)

    # 2. Calculate the log of those probabilities
    log_probs = torch.log(softmax)

    # 3. Select the log probability of the correct class for each example in the batch.
    batch_size = y.shape[0]
    actual_log_probs = log_probs[range(batch_size), y]

    # 4. Compute the mean of the negative log probabilities (cross-entropy loss)
    J = -actual_log_probs.mean()

    #--------------------------  Compute gradients------------- (mathematic word de)
    # Calculate the gradient of the loss
    # Initialize gradient tensor with probabilities
    dJdy_hat = softmax.clone()
    # Subtract 1 from the true class probabilities (where y_j = 1)
    dJdy_hat[range(batch_size), y] -= 1
    # Average the gradients over the batch
    dJdy_hat /= batch_size
    return J, dJdy_hat





# modelleme

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomLinearLayer:
    def __init__(self, in_features, out_features):
        self.in_features=in_features
        self.out_features=out_features        
        self.weight = torch.randn(out_features, in_features, requires_grad=True)#default
        self.bias = torch.randn(out_features, requires_grad=True)#default
        self.grad_weight = torch.zeros(out_features, in_features) #placeholder for grad weight
        self.grad_bias = torch.zeros(out_features) #placeholder for grad bias
        self.input=None#placeholder,for storing x value
        
    def forward(self, x):
        self.input = x
        return self.input @ self.weight.t() + self.bias

    def backward(self, grad_output):
        # grad_output is the gradient of the loss with respect to the output of the linear layer
        
        # Calculate the gradients of the weight (dL/dW)
        # This is the dot product of the input transposed (x^T) and the gradient of the loss with respect to the output of the layer (dL/dy)  
        # grad_weight is computed using torch.mm for matrix multiplication between the transposed gradient output and the input to the layer.
        self.grad_weight.data = grad_output.t() @ self.input  # Gradient with respect to the weight
        
        # Calculate the gradients of the biases (dL/db)
        # This is the sum of the gradients for each example in the batch
        # grad_bias is computed by summing grad_output across the batch dimension.
        self.grad_bias.data = grad_output.sum(0)  # Gradient with respect to the bias

        # Calculate the gradient of the loss with respect to the input of the layer (dL/dx)
        # This is needed for layers that come before this one in the network
        #  This will be used by the previous layer in the network for its backward pass.
        grad_input = grad_output @ self.weight

        # Update weight and biases here if doing so manually, or return gradients for use in an optimizer
        # self.weight.data -= learning_rate * grad_weight
        # self.bias.data -= learning_rate * grad_bias
        
        return grad_input# grad_input

In [5]:
from activations import BaseActivationFunction,SigmoidActivationFunction, ReLUActivationFunction, IdentityActivationFunction
from collections import OrderedDict


In [6]:

class CustomSequentialNetwork:
    def __init__(self,layers):
        # Using OrderedDict to maintain the layers in order
        self.layers = layers

    def forward(self, x):
        self.forward_results = []
        input = x
        for name, layer in self.layers.items():
            input = layer.forward(input)  
            self.forward_results.append((name, input))
        return self.forward_results[-1][1]

    def backward(self, loss_grad):
        # Start with the gradient of the loss
        grad = loss_grad
        # Iterate over the layers in reverse order
        for name, layer in reversed(self.layers.items()):
            if isinstance(layer, CustomLinearLayer):
                grad = layer.backward(grad)
            if isinstance(layer, BaseActivationFunction):
                grad = grad*layer.backward() 
    
    def __repr__(self):
        lines = ['CustomSequentialNetwork(']
        for name, layer in self.layers.items():
            # For custom layers
            if isinstance(layer, CustomLinearLayer):
                lines.append(f'  ({name}): {layer.__class__.__name__}('
                             f'in_features={layer.in_features}, out_features={layer.out_features})')
            # For activation functions or other types of layers
            else:
                # For simplicity, we assume that non-custom layers are represented by their class name as strings
                lines.append(f'  ({name}): {layer}')
        lines.append(')')
        return '\n'.join(lines)

    def show_forward_result(self):
        print(self.forward_results)

In [7]:
torch.manual_seed(0)

# Example usage of the custom linear layer

input_tensor = torch.randn(10, 2,requires_grad=True)
x=input_tensor.clone()
desired_output_tensor = torch.randint(0,5,(10,))

net=CustomSequentialNetwork(
    OrderedDict([
            ('linear1', CustomLinearLayer(2, 20)),
            ('relu', ReLUActivationFunction()),
            ('linear2', CustomLinearLayer(20, 5)),
            #('sigmoid', SigmoidActivationFunction()),

    ])  
)
net=CustomSequentialNetwork(
    OrderedDict([
            ('linear1', CustomLinearLayer(2, 20)),
            ('sigmoid1', SigmoidActivationFunction()),
            ('linear2', CustomLinearLayer(20, 5)),
            ('sigmoid2', SigmoidActivationFunction()),
            ('sigmoid3', SigmoidActivationFunction()),

    ])  
)
print(net)
output_custom = net.forward(input_tensor)
print(f" output_custom , {output_custom} \n")
#J, dJdy_hat = mse_loss(desired_output_tensor, output_custom)
J, dJdy_hat = cross_entropy_loss(desired_output_tensor, output_custom)
net.backward(dJdy_hat)

CustomSequentialNetwork(
  (linear1): CustomLinearLayer(in_features=2, out_features=20)
  (sigmoid1): <activations.SigmoidActivationFunction object at 0x7f0296185650>
  (linear2): CustomLinearLayer(in_features=20, out_features=5)
  (sigmoid2): <activations.SigmoidActivationFunction object at 0x7f0296186590>
  (sigmoid3): <activations.SigmoidActivationFunction object at 0x7f0296185990>
)
 output_custom , tensor([[0.7298, 0.5026, 0.5960, 0.6486, 0.5384],
        [0.7291, 0.5087, 0.5522, 0.6580, 0.5239],
        [0.7276, 0.5032, 0.5772, 0.6579, 0.5276],
        [0.7290, 0.5768, 0.5714, 0.6752, 0.5095],
        [0.7291, 0.5203, 0.5462, 0.6712, 0.5186],
        [0.7293, 0.5531, 0.5814, 0.6986, 0.5121],
        [0.7296, 0.5974, 0.6158, 0.7058, 0.5081],
        [0.7285, 0.5096, 0.5516, 0.6711, 0.5234],
        [0.7295, 0.5769, 0.5870, 0.6965, 0.5092],
        [0.7290, 0.5543, 0.5978, 0.7055, 0.5119]], grad_fn=<MulBackward0>) 



In [8]:
torch.manual_seed(0)

# compare the result with autograd
net_autograd = nn.Sequential(
    OrderedDict([
        ('linear1', nn.Linear(2, 20)),
        ('relu', nn.ReLU()),
        ('linear2', nn.Linear(20, 5)),
        #('sigmoid', nn.Sigmoid()),

    ])
)
net_autograd = nn.Sequential(
    OrderedDict([
        ('linear1', nn.Linear(2, 20)),
        ('sigmoid1', nn.Sigmoid()),
        ('linear2', nn.Linear(20, 5)),
        ('sigmoid2', nn.Sigmoid()),
        ('sigmoid3', nn.Sigmoid()),

    ])
)
print(net_autograd)
net_autograd.linear1.weight.data = net.layers["linear1"].weight
net_autograd.linear1.bias.data = net.layers["linear1"].bias
net_autograd.linear2.weight.data = net.layers["linear2"].weight
net_autograd.linear2.bias.data = net.layers["linear2"].bias

y_hat_autograd = net_autograd(x)
print(y_hat_autograd)
#J_autograd = F.mse_loss(y_hat_autograd, desired_output_tensor)
J_autograd = torch.nn.CrossEntropyLoss()(y_hat_autograd, desired_output_tensor)
net_autograd.zero_grad()
J_autograd.backward()

Sequential(
  (linear1): Linear(in_features=2, out_features=20, bias=True)
  (sigmoid1): Sigmoid()
  (linear2): Linear(in_features=20, out_features=5, bias=True)
  (sigmoid2): Sigmoid()
  (sigmoid3): Sigmoid()
)
tensor([[0.7298, 0.5026, 0.5960, 0.6486, 0.5384],
        [0.7291, 0.5087, 0.5522, 0.6580, 0.5239],
        [0.7276, 0.5032, 0.5772, 0.6579, 0.5276],
        [0.7290, 0.5768, 0.5714, 0.6752, 0.5095],
        [0.7291, 0.5203, 0.5462, 0.6712, 0.5186],
        [0.7293, 0.5531, 0.5814, 0.6986, 0.5121],
        [0.7296, 0.5974, 0.6158, 0.7058, 0.5081],
        [0.7285, 0.5096, 0.5516, 0.6711, 0.5234],
        [0.7295, 0.5769, 0.5870, 0.6965, 0.5092],
        [0.7290, 0.5543, 0.5978, 0.7055, 0.5119]], grad_fn=<SigmoidBackward0>)


In [9]:
print((net_autograd.linear1.weight.grad.data -  net.layers["linear1"].grad_weight).norm() < 1e-3)
print((net_autograd.linear1.bias.grad.data -  net.layers["linear1"].grad_bias).norm() < 1e-3)
print((net_autograd.linear2.weight.grad.data -  net.layers["linear2"].grad_weight).norm() < 1e-3)
print((net_autograd.linear2.bias.grad.data - net.layers["linear2"].grad_bias).norm()< 1e-3)

tensor(True)
tensor(True)
tensor(True)
tensor(True)
